# Demo 07g — Ensembles on Real Tables

**Class 07 · E7** (slides: `slides/07_ensembles.md`)

The earlier demos use generated data to show where each model breaks. This one asks the applied question: on real tables, at full size, with each model tuned, which family wins?

* Five classification tasks and six regression tasks from `datasets/`, all rows.
* 5-fold cross-validation, the same folds for every model. Tuning uses the training folds only.
* Naive Bayes classifies only: its column holds Bayesian ridge on the regression tables.
* Boosting uses early stopping. Stacking uses strong, different members: an SVM, a forest, boosting, k-NN and a linear model.

Results are kept in `.cache/07_tables.csv`: delete that file to run them again (several minutes on 16 cores).

In [ ]:
import warnings

import numpy as np
import polars as pl
from joblib import Parallel, delayed

from faa import boundaries, data, demo, metrics, plotlib
from faa import galleries as ga
from faa.demo import show, slide
from faa.galleries import COLOUR
from faa.plotlib import RED

demo.setup("07")
SEED = 0

In [ ]:
import os  # noqa: E402
import time  # noqa: E402

import optuna  # noqa: E402
from scipy import stats  # noqa: E402
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin, clone  # noqa: E402
from sklearn.calibration import CalibratedClassifierCV  # noqa: E402
from sklearn.ensemble import (  # noqa: E402
    AdaBoostClassifier,
    AdaBoostRegressor,
    BaggingClassifier,
    BaggingRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
    StackingClassifier,
    StackingRegressor,
    VotingClassifier,
    VotingRegressor,
)
from sklearn.linear_model import BayesianRidge, LogisticRegression, Ridge  # noqa: E402
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score  # noqa: E402
from sklearn.naive_bayes import GaussianNB  # noqa: E402
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor  # noqa: E402
from sklearn.neural_network import MLPClassifier, MLPRegressor  # noqa: E402
from sklearn.pipeline import make_pipeline  # noqa: E402
from sklearn.preprocessing import StandardScaler  # noqa: E402
from sklearn.svm import SVC, SVR  # noqa: E402
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor  # noqa: E402
from threadpoolctl import threadpool_limits  # noqa: E402
from xgboost import XGBClassifier, XGBRegressor  # noqa: E402

TABLES = data.TABLES
FAMILY_OF = {
    "linear": "shallow", "naive Bayes": "shallow", "tree": "shallow", "1-NN": "shallow", "SVM": "shallow", "MLP": "shallow",
    "soft vote": "voting", "weighted vote": "voting", "k-NN": "voting",
    "bagged trees": "bagging", "random forest": "bagging",
    "AdaBoost": "boosting", "HistGB": "boosting", "XGBoost": "boosting",
    "stacking": "meta", "neural stack": "meta",
}  # fmt: skip
MODELS = list(FAMILY_OF)


def suggest(trial, name, spec):
    """A list is a choice, ("log", a, b) a real value on a log scale, ("int", a, b) an integer."""
    if isinstance(spec, list):
        return trial.suggest_categorical(name, spec)
    return (
        trial.suggest_float(name, spec[1], spec[2], log=True)
        if spec[0] == "log"
        else trial.suggest_int(name, spec[1], spec[2])
    )


class Tuned(BaseEstimator):
    """Optuna (TPE, 20 trials) with 3-fold CV on at most 2000 training rows, then a refit of the best setting on all of them."""

    def __init__(self, model=None, grid=None, task="c", seed=0):
        self.model, self.grid, self.task, self.seed = model, grid, task, seed

    def fit(self, X, y):
        rows = np.random.default_rng(self.seed).permutation(len(y))[:2000]
        cv = (StratifiedKFold if self.task == "c" else KFold)(3, shuffle=True, random_state=self.seed)
        scoring = "matthews_corrcoef" if self.task == "c" else "neg_mean_absolute_error"

        def objective(trial):
            model = clone(self.model).set_params(**{k: suggest(trial, k, v) for k, v in self.grid.items()})
            return cross_val_score(model, X[rows], y[rows], cv=cv, scoring=scoring).mean()

        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=self.seed))
        study.optimize(objective, n_trials=20)
        self.best_ = clone(self.model).set_params(**study.best_params).fit(X, y)
        self.classes_ = getattr(self.best_, "classes_", None)
        return self

    def predict(self, X):
        return self.best_.predict(X)

    def predict_proba(self, X):
        return self.best_.predict_proba(X)


class TunedClassifier(ClassifierMixin, Tuned):
    pass


class TunedRegressor(RegressorMixin, Tuned):
    pass


class TunedStack(BaseEstimator):
    """Stacking in practice: tune each member once on the training set, then stack the members with those settings."""

    def __init__(self, members=None, task="c"):
        self.members, self.task = members, task

    def fit(self, X, y):
        fixed = [(n, clone(m.fit(X, y).best_) if isinstance(m, Tuned) else m) for n, m in self.members]
        c = self.task == "c"
        folds = (StratifiedKFold if c else KFold)(
            3, shuffle=True, random_state=0
        )  # the tables are stored in order: shuffle
        stack = (
            StackingClassifier(fixed, final_estimator=LogisticRegression(max_iter=2000), cv=folds)
            if c
            else StackingRegressor(fixed, final_estimator=Ridge(), cv=folds)
        )
        self.model_ = stack.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X)


def table_model(name, task, seed):
    c = task == "c"
    tuned = lambda m, grid: (TunedClassifier if c else TunedRegressor)(m, grid, task, seed)  # noqa: E731
    scaled = lambda m: make_pipeline(StandardScaler(), m)  # noqa: E731
    svm = scaled(CalibratedClassifierCV(SVC(), method="sigmoid", cv=3, ensemble=False) if c else SVR())
    svm_grid = {
        f"{'calibratedclassifiercv__estimator' if c else 'svr'}__{k}": v
        for k, v in {"C": ("log", 0.1, 300), "gamma": ("log", 1e-4, 1)}.items()
    }
    knn = scaled((KNeighborsClassifier if c else KNeighborsRegressor)(weights="distance"))
    knn_grid = {f"kneighbors{'classifier' if c else 'regressor'}__n_neighbors": ("int", 3, 60)}
    rf = (RandomForestClassifier if c else RandomForestRegressor)(n_estimators=300, random_state=seed)
    hgb = (HistGradientBoostingClassifier if c else HistGradientBoostingRegressor)(
        max_iter=1000,
        learning_rate=0.05,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=seed,
    )
    hgb_grid = {
        "max_leaf_nodes": ("int", 7, 63),
        "min_samples_leaf": ("int", 5, 60),
        "l2_regularization": ("log", 1e-3, 10),
        "learning_rate": ("log", 0.02, 0.2),
    }
    linear = scaled(LogisticRegression(max_iter=2000) if c else Ridge())
    members = [
        ("svm", tuned(svm, svm_grid)),
        ("forest", rf),
        ("boosting", tuned(hgb, hgb_grid)),
        ("knn", tuned(knn, knn_grid)),
        ("linear", linear),
    ]
    shallow_vote = [("linear", linear), ("tree", tuned((DecisionTreeClassifier if c else DecisionTreeRegressor)(random_state=seed), {"max_depth": ("int", 3, 14), "min_samples_leaf": ("int", 1, 40)})),
                    ("svm", tuned(svm, svm_grid)), ("knn", tuned(knn, knn_grid))]  # fmt: skip
    xgb = (XGBClassifier if c else XGBRegressor)(
        n_estimators=500, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=seed, n_jobs=1
    )
    return {
        "linear": linear,
        "naive Bayes": GaussianNB() if c else scaled(BayesianRidge()),
        "tree": shallow_vote[1][1],
        "1-NN": scaled((KNeighborsClassifier if c else KNeighborsRegressor)(n_neighbors=1)),
        "SVM": tuned(svm, svm_grid),
        "MLP": scaled(
            (MLPClassifier if c else MLPRegressor)(
                hidden_layer_sizes=(64,), early_stopping=True, max_iter=800, random_state=seed
            )
        ),
        "soft vote": VotingClassifier(shallow_vote, voting="soft") if c else VotingRegressor(shallow_vote),
        "weighted vote": ga.WeightedVote(dict(shallow_vote), task),
        "k-NN": tuned(knn, knn_grid),
        "bagged trees": (
            BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, random_state=seed)
            if c
            else BaggingRegressor(DecisionTreeRegressor(), n_estimators=200, random_state=seed)
        ),
        "random forest": rf,
        "AdaBoost": (
            AdaBoostClassifier(DecisionTreeClassifier(max_depth=3), n_estimators=300, random_state=seed)
            if c
            else AdaBoostRegressor(DecisionTreeRegressor(max_depth=5), n_estimators=300, random_state=seed)
        ),
        "HistGB": tuned(hgb, hgb_grid),
        "XGBoost": tuned(
            xgb,
            {
                "max_depth": ("int", 3, 9),
                "min_child_weight": ("log", 0.5, 20),
                "reg_lambda": ("log", 0.1, 20),
                "learning_rate": ("log", 0.01, 0.2),
            },
        ),
        "stacking": TunedStack(members, task),
    }[name]


TOPOLOGY = ([(256, "tanh"), (64, "tanh")], [(128, "swish"), (128, "swish"), (32, "swish")])


def neural_fit_predict(X_tr, y_tr, X_te, task, seed):
    """The neural stack of Demo 07e: two networks, frozen, and a head on their last hidden layers."""
    os.environ["KERAS_BACKEND"] = "jax"
    os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
    import keras
    from keras import layers

    keras.utils.set_random_seed(seed)
    scaler = StandardScaler().fit(X_tr)
    X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)
    mean, sd = (float(y_tr.mean()), float(y_tr.std())) if task == "r" else (0.0, 1.0)
    t = (y_tr - mean) / sd
    order = np.random.default_rng(seed).permutation(len(t))
    cut = int(0.8 * len(t))
    fit, val = order[:cut], order[cut:]
    activation, loss = ("sigmoid", "binary_crossentropy") if task == "c" else (None, "mse")

    def train(model):
        model.compile(keras.optimizers.Adam(2e-3), loss)
        stop = keras.callbacks.EarlyStopping("val_loss", patience=30, restore_best_weights=True)
        model.fit(
            X_tr[fit],
            t[fit],
            validation_data=(X_tr[val], t[val]),
            epochs=200,
            batch_size=64,
            verbose=0,
            callbacks=[stop],
        )

    bodies = []
    for hidden_layers in TOPOLOGY:
        inputs = keras.Input((X_tr.shape[1],))
        h = inputs
        for units, act in hidden_layers:
            h = layers.Dense(units, activation=act, kernel_regularizer=keras.regularizers.L2(1e-5))(h)
        train(keras.Model(inputs, layers.Dense(1, activation=activation)(h)))
        body = keras.Model(inputs, h)
        body.trainable = False
        bodies.append(body)
    inputs = keras.Input((X_tr.shape[1],))
    h = layers.Concatenate()([b(inputs) for b in bodies])
    h = layers.Dense(32, activation="relu", kernel_regularizer=keras.regularizers.L2(1e-3))(h)
    head = keras.Model(inputs, layers.Dense(1, activation=activation)(h))
    train(head)
    out = head.predict(X_te, verbose=0, batch_size=4096).ravel()
    return (out > 0.5).astype(int) if task == "c" else out * sd + mean


def table_fold(dataset, name, fold, tr, te):
    X, y = TABLES[dataset][0]()
    task = TABLES[dataset][1]
    with threadpool_limits(1), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.perf_counter()
        if name == "neural stack":
            pred = neural_fit_predict(X[tr], y[tr], X[te], task, fold)
        else:
            pred = table_model(name, task, fold).fit(X[tr], y[tr]).predict(X[te])
    return dataset, name, fold, metrics.task_score(task, y[te], pred), time.perf_counter() - t0

In [ ]:
slide("Real Tables: the Board")
CACHE = plotlib.ROOT / ".cache" / "07_tables.csv"
if CACHE.exists():
    results = pl.read_csv(CACHE)
else:
    jobs = []
    for d, (load, task) in TABLES.items():
        X, y = load()
        cv = (
            StratifiedKFold(5, shuffle=True, random_state=SEED)
            if task == "c"
            else KFold(5, shuffle=True, random_state=SEED)
        )
        for fold, (tr, te) in enumerate(cv.split(X, y)):
            jobs += [(d, m, fold, tr, te) for m in MODELS]
    done = Parallel(n_jobs=-1, batch_size=1)(delayed(table_fold)(*j) for j in jobs)
    results = pl.DataFrame(done, schema=["dataset", "model", "seed", "score", "seconds"], orient="row")
    CACHE.parent.mkdir(exist_ok=True)
    results.write_csv(CACHE)
class_sets = [d for d, (_, t) in TABLES.items() if t == "c"]
reg_sets = [d for d, (_, t) in TABLES.items() if t == "r"]
show(ga.wide(results.drop("seconds"), MODELS, class_sets))
show(ga.wide(results.drop("seconds"), MODELS, reg_sets))

In [ ]:
slide("Real Tables: Mean Rank")
means = results.group_by("dataset", "model").agg(pl.col("score").mean())
ranks = pl.concat(
    [
        means.filter(pl.col("dataset") == d).with_columns(
            pl.col("score").rank("average", descending=task == "c").alias("rank")
        )
        for d, (_, task) in TABLES.items()
    ]
)
winners = (
    ranks.filter(pl.col("rank") == pl.col("rank").min().over("dataset"))
    .select("dataset", "model", "score")
    .sort("dataset")
)
show(winners)
rank_table = ranks.group_by("model").agg(pl.col("rank").mean()).sort("rank")
rank_table = rank_table.with_columns(
    pl.col("model").replace_strict(FAMILY_OF).alias("family"),
    pl.col("model")
    .map_elements(lambda m: winners["model"].to_list().count(m), return_dtype=pl.Int64)
    .alias("tables won"),
)
show(rank_table, 2)
wide_rank = ranks.pivot(on="model", index="dataset", values="rank").select(MODELS).to_numpy()
n_tables, n_models = wide_rank.shape
critical = (
    stats.studentized_range.ppf(0.95, n_models, np.inf)
    / np.sqrt(2)
    * np.sqrt(n_models * (n_models + 1) / (6 * n_tables))
)
print(
    f"Friedman test over {n_tables} tables, {n_models} models: p = {stats.friedmanchisquare(*wide_rank.T).pvalue:.2e}"
)
print(f"Nemenyi critical difference at 5 %: {critical:.2f} ranks")
best_rank = rank_table["rank"][0]
print("clearly behind the best model:", rank_table.filter(pl.col("rank") > best_rank + critical)["model"].to_list())

In [ ]:
slide("Real Tables: Best Family per Table")
rows = []
for d, (_, task) in TABLES.items():
    here = means.filter(pl.col("dataset") == d).with_columns(pl.col("model").replace_strict(FAMILY_OF).alias("family"))
    best = here.group_by("family").agg(pl.col("score").max() if task == "c" else pl.col("score").min())
    values = dict(zip(best["family"], best["score"]))
    base = values["shallow"]
    rows += [
        (d, f, 100 * (v - base) / abs(base) * (1 if task == "c" else -1)) for f, v in values.items() if f != "shallow"
    ]
family_gain = pl.DataFrame(rows, schema=["dataset", "family", "gain %"], orient="row")
gain_wide = family_gain.pivot(on="family", index="dataset", values="gain %")
show(gain_wide.select("dataset", "voting", "bagging", "boosting", "meta"), 1)
print(
    "median gain over the best shallow model, %:",
    family_gain.group_by("family").agg(pl.col("gain %").median()).sort("gain %", descending=True).rows(),
)

In [ ]:
# plot e7_rank
fg, ax = plotlib.fig("full", 2.05)
y = np.arange(len(rank_table))[::-1]
ax.barh(y, rank_table["rank"], color=[COLOUR[f] for f in rank_table["family"]], height=0.68)
ax.axvline(best_rank + critical, color=RED, lw=0.7, ls=":")
for yi, r, n in zip(y, rank_table["rank"], rank_table["tables won"]):
    ax.text(r + 0.12, yi, f"{r:.1f}" + (f"  won {n}" if n else ""), fontsize=6, va="center")
ax.set_yticks(y, rank_table["model"].to_list(), fontsize=6.2)
ax.set(xlabel="mean rank over 11 tables (1 is best)", xlim=(0, 16))
ax.grid(axis="y", visible=False)
plotlib.save(fg, "e7_rank")

In [ ]:
def real_board(sets, higher):
    colours = {m: COLOUR[f] for m, f in FAMILY_OF.items()}
    values = {(d, m): v for d, m, v in means.iter_rows()}
    return plotlib.score_board(values, sets, MODELS, colours, higher=higher, left=0.18)


plotlib.save(real_board(class_sets, True), "e7_board_c")

In [ ]:
plotlib.save(real_board(reg_sets, False), "e7_board_r")

In [ ]:
# plot e7_family
fg, ax = plotlib.fig("full", 1.65)
fams = ["voting", "bagging", "boosting", "meta"]
sets = gain_wide["dataset"].to_list()
for k, f in enumerate(fams):
    ax.bar(np.arange(len(sets)) + (k - 1.5) * 0.2, gain_wide[f].clip(-10, 40), width=0.19, color=COLOUR[f], label=f)
ax.axhline(0, color=boundaries.LINE, lw=0.6)
ax.set_xticks(
    np.arange(len(sets)),
    [d.replace(", good or not", ", binary").replace(" housing", "") for d in sets],
    fontsize=5.6,
    rotation=90,
)
ax.set(ylabel="gain over best shallow, %")
ax.grid(axis="x", visible=False)
ax.legend(ncols=4, loc="upper right", fontsize=6.3, handlelength=1.1, columnspacing=0.8)
plotlib.save(fg, "e7_family")